<a href="https://colab.research.google.com/github/anelchik/anelsinternshipwork/blob/main/w02_ml_task_framing_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: ranking / scoring.**

My lane is a freestyle content-review prioritization problem. The useful decision is not just "is this page bad?" The useful decision is: **which pages should an SEO/content team review first?**

The model output would be a priority score for each content item. Higher score means the page has stronger evidence of search-intent mismatch or content underperformance and should be reviewed earlier.

I am choosing ranking/scoring instead of plain classification because editors usually have limited time. They do not need a perfect yes/no label for every page; they need a ranked queue of the most suspicious pages.

In [ ]:
task_type = "ranking / scoring"
output = "content_review_priority_score"
unit = "one content item/page"

print(f"Task type: {task_type}")
print(f"Model output: {output}")
print(f"Unit of analysis: {unit}")

Task type: ranking / scoring
Model output: content_review_priority_score
Unit of analysis: one content item/page


## 2. Target or proxy

The true target I care about is **search-intent mismatch**: pages where users arrive with expectations that the content does not satisfy.

That target is not directly observed in the starter data. I should not pretend that I have a clean human label for it.

So my first proxy target is a **review-priority score** based on observed signals:

- the page has search demand, such as impressions or clicks;
- users do not respond strongly, such as low CTR or low engagement;
- the page shows weak or declining performance.

For later warehouse work, a better observed proxy would be **future decline after a safe feature window**, for example whether clicks or impressions drop in the next 30 days. That would be closer to an observed outcome. Until then, I will call this a proxy, not ground truth.

In [ ]:
target_or_proxy = {
    "true_target": "search-intent mismatch",
    "available_now": "proxy score from observable search and engagement signals",
    "better_later_target": "future 30-day decline after a safe feature window",
    "claim_limit": "decision-support proxy, not ground-truth proof"
}

for key, value in target_or_proxy.items():
    print(f"{key}: {value}")

true_target: search-intent mismatch
available_now: proxy score from observable search and engagement signals
better_later_target: future 30-day decline after a safe feature window
claim_limit: decision-support proxy, not ground-truth proof


## 3. Success metric

My main metric is **Precision@K** for a ranked review queue.

This means: if the model recommends the top K pages for review, what fraction of those pages are actually useful review candidates according to the proxy or later observed outcome?

A defensible first goal is:

> **Precision@50 should beat a simple baseline rule** such as ranking pages only by impressions or only by declining label.

This metric fits the real action because an editor can only review a limited number of pages. A model that is useful at the top of the queue is more valuable than a model that looks good only on average.

In [ ]:
success_metric = "Precision@K"
k = 50
baseline = "simple rule: rank by impressions or current decline label"

print(f"Metric: {success_metric}")
print(f"K: {k}")
print(f"Good means: top-{k} recommendations beat the baseline queue")
print(f"Baseline: {baseline}")

Metric: Precision@K
K: 50
Good means: top-50 recommendations beat the baseline queue
Baseline: simple rule: rank by impressions or current decline label


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one content item/page**.

Each row should represent one pseudonymized content item, not one client and not one keyword. The dataframe below tries to load the starter CSV from the normal repo path. In this sandbox, if the CSV is not present, the cell creates a tiny example with the same intended grain so the notebook still runs top to bottom.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Normal repo path for the FlyRank starter dataset.
possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("/mnt/data/content_refresh_anonymized.csv"),
]

csv_path = next((p for p in possible_paths if p.exists()), None)

if csv_path is not None:
    df = pd.read_csv(csv_path)
    source = str(csv_path)
else:
    # Fallback only for this chat sandbox where the repo dataset was not uploaded.
    # In the actual FlyRank repo, the cell should load data/raw/content_refresh_anonymized.csv.
    df = pd.DataFrame({
        "content_id": ["content_001", "content_002", "content_003", "content_004", "content_005"],
        "client_id": ["client_a", "client_a", "client_b", "client_b", "client_c"],
        "content_type": ["blog", "landing_page", "blog", "guide", "landing_page"],
        "impressions_90d": [12500, 8700, 4300, 23000, 6100],
        "clicks_90d": [60, 95, 18, 210, 24],
        "ctr": [0.48, 1.09, 0.42, 0.91, 0.39],
        "avg_position": [14.2, 8.7, 20.5, 11.3, 18.9],
        "engagement_rate": [21.0, 47.5, 18.2, 35.1, 16.4],
        "is_declining_label": [1, 0, 1, 0, 1],
    })
    source = "fallback example because starter CSV was not uploaded here"

print(f"Loaded source: {source}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

# Build a safe lane slice. Do NOT use trend_pct or trend_direction as features.
preferred_cols = [
    "content_id", "client_id", "content_type", "impressions_90d", "clicks_90d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "word_count", "is_declining_label"
]
existing_cols = [c for c in preferred_cols if c in df.columns]
lane_df = df[existing_cols].copy()

# Add a simple interpretable proxy score for initial framing only.
# Higher score = more search demand + weaker response.
if {"impressions_90d", "ctr"}.issubset(lane_df.columns):
    impressions_rank = lane_df["impressions_90d"].rank(pct=True)
    weak_ctr_rank = (-lane_df["ctr"]).rank(pct=True)
    lane_df["rough_review_priority_score"] = (0.6 * impressions_rank + 0.4 * weak_ctr_rank).round(3)

print("One row = one pseudonymized content item/page")
display(lane_df.head(10))

Loaded source: fallback example because starter CSV was not uploaded here
Rows: 5
Columns: 9
One row = one pseudonymized content item/page


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,is_declining_label,rough_review_priority_score
0,content_001,client_a,blog,12500,60,0.48,14.2,21.0,1,0.72
1,content_002,client_a,landing_page,8700,95,1.09,8.7,47.5,0,0.44
2,content_003,client_b,blog,4300,18,0.42,20.5,18.2,1,0.44
3,content_004,client_b,guide,23000,210,0.91,11.3,35.1,0,0.76
4,content_005,client_c,landing_page,6100,24,0.39,18.9,16.4,1,0.64


## 5. Why ML beats a fixed rule here

A fixed rule like "review pages with low CTR" is too weak because the problem depends on several signals at the same time.

For example, low CTR can mean different things depending on impressions, average position, content type, engagement, scroll behavior, and whether the page is already declining. A page with low CTR but almost no impressions may not be worth reviewing. A page with decent CTR but very poor engagement may still be a mismatch. A page in position 2 and a page in position 40 should not be judged by the same rule.

ML or learned scoring can help because it can combine many imperfect signals into one ranked queue. The final claim will still be careful: the score supports review decisions; it does not prove the exact cause of decline or prove what users intended.

In [ ]:
why_ml = [
    "The action is top-K prioritization, not a yes/no decision for every page.",
    "Signals interact: impressions, CTR, average position, content type, and engagement mean different things together.",
    "A single threshold wastes review time on noisy cases.",
    "The model should support editor decisions, not replace human review or claim causality."
]

for i, reason in enumerate(why_ml, 1):
    print(f"{i}. {reason}")

1. The action is top-K prioritization, not a yes/no decision for every page.
2. Signals interact: impressions, CTR, average position, content type, and engagement mean different things together.
3. A single threshold wastes review time on noisy cases.
4. The model should support editor decisions, not replace human review or claim causality.


## One-paragraph frame

For SEO specialists and content editors, deciding **which pages to review first**, I will build a **ranking/scoring output** from pseudonymized content performance data. The score will prioritize pages with evidence of search demand but weak user response, using a proxy now and later testing against observed future decline. Success will be measured with **Precision@K**, especially whether the top review queue beats a simple baseline rule. A wrong recommendation costs wasted editor time and missed high-impact pages. A plain rule is not enough because the signals are noisy, interacting, and different across content types and clients. I will claim only decision-support and directional evidence, not causal proof.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Ready to commit under `work/notebooks/w02_ml_task_framing.ipynb` and submit the public repo URL on the card.